# TradoVera — Tutoriel 3 : Feature Engineering pour le Machine Learning

Ce notebook montre comment générer un dataset consolidé unique (prix daily + volatilité réalisée + PE Ratio mensuel + rapports trimestriels) en utilisant le SDK, pour entraîner des modèles de prédiction.

In [ ]:
import os
import sys

import numpy as np

sys.path.append(os.path.abspath(".."))
import tvdata_sdk

### 1. Générer le dataset consolidé brut
Appelons la fonction `get_consolidated_dataset()` pour assembler toutes les dimensions de données en une seule table temporelle quotidienne :

In [ ]:
df_raw = tvdata_sdk.get_consolidated_dataset("AAPL")
print(f"Dataset chargé : {len(df_raw)} lignes, {len(df_raw.columns)} colonnes.")
df_raw.tail(10)

### 2. Traitement des valeurs manquantes (Forward-Fill)
Pour entraîner un modèle, il est utile de propager les dernières valeurs connues des fondamentaux mensuels et trimestriels pour combler les valeurs NaN :

In [ ]:
# Colonnes à propager
cols_to_fill = [
    "pe_ratio",
    "price_to_book",
    "beta_raw",
    "realized_vol_30d",
    "revenue",
    "net_income",
    "eps",
]

df_filled = df_raw.copy()
df_filled[cols_to_fill] = df_filled[cols_to_fill].ffill()

# Afficher le dataset nettoyé sans valeurs manquantes
df_filled = df_filled.dropna(subset=["close"]).reset_index(drop=True)
df_filled.tail(10)

### 3. Calculer les caractéristiques (Features) cibles
Créons quelques features classiques pour notre modèle (RSI, Rendements, Ecart de PE...) :

In [ ]:
# Target : Le cours va-t-il monter le lendemain ?
df_filled["Target"] = (df_filled["close"].shift(-1) > df_filled["close"]).astype(int)

# Feature 1 : Rendement historique 5 jours
df_filled["Return_5D"] = df_filled["close"].pct_change(5)

# Feature 2 : Logarithme du PE ratio
df_filled["Log_PE"] = np.log(df_filled["pe_ratio"].astype(float) + 1.0)

# Feature 3 : Volatilité historique vs Volatilité Bloomberg
df_filled["Vol_Ratio"] = (
    df_filled["realized_vol_30d"] / df_filled["realized_vol_30d"].rolling(20).mean()
)

df_filled[["date", "close", "pe_ratio", "Target", "Return_5D", "Vol_Ratio"]].tail(5)